# 타이타닉 생존자 예측

## 목차
1. 데이터 로드 및 기본 정보 확인
2. 데이터 전처리
3. 모델 학습 및 평가
4. 예측 및 제출
5. 하이퍼파라미터 튜닝

## 1. 데이터 로드 및 기본 정보 확인

In [14]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

In [67]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [38]:
test_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [ ]:
#	PassengerId
# Survived	         생존 여부 (0: 사망, 1: 생존)
# Pclass	           좌석 등급
# Name	Sex	Age
# SibSp	             함께 탑승한 형제, 자매 또는 배우자의 수
# Parch              함께 탑승한 부모 또는 자녀의 수
# Ticket
# Fare	             요금 (탑승 비용)
# Cabin	             선실 번호
# Embarked           탑승한 항구 (C: Cherbourg, Q: Queenstown, S: Southampton)

## 2. 데이터 전처리

In [15]:
# 데이터 전처리 함수
import re
def preprocess(df):
    df = df.copy()

    # def normalize_name(name):
    #     # 이름에서 특수문자를 제거하고 정규화
    #     return re.sub(r"[^a-zA-Z\s]", "", name)

    def ticket_number(ticket):
        numbers = re.findall(r"\d+", ticket)  # 모든 숫자 패턴 찾기
        return "".join(numbers)  # 리스트를 문자열로 변환

    def ticket_item(ticket):
        # 티켓 종류를 추출하는 코드
        items = ticket.split(" ")
        if len(items) == 1:
            return "NONE"
        return items[0]

    # 전처리 함수들을 데이터프레임에 적용
    # df["Name"] = df["Name"].apply(normalize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    df["Ticket_item"] = df["Ticket"].apply(ticket_item)

    return df

In [16]:
# 전처리 함수 적용
preprocessed_train_df = preprocess(train_df)
preprocessed_test_df = preprocess(test_df)

preprocessed_train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Ticket_number,Ticket_item
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,521171,A/5
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,17599,PC
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,23101282,STON/O2.
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,113803,NONE
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,373450,NONE


In [5]:
# 기초 통계량
preprocessed_train_df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [17]:
preprocessed_train_df.info() #  Cabin, Age에 결측치 많음, Embarked 조금

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PassengerId    891 non-null    int64  
 1   Survived       891 non-null    int64  
 2   Pclass         891 non-null    int64  
 3   Name           891 non-null    object 
 4   Sex            891 non-null    object 
 5   Age            714 non-null    float64
 6   SibSp          891 non-null    int64  
 7   Parch          891 non-null    int64  
 8   Ticket         891 non-null    object 
 9   Fare           891 non-null    float64
 10  Cabin          204 non-null    object 
 11  Embarked       889 non-null    object 
 12  Ticket_number  891 non-null    object 
 13  Ticket_item    891 non-null    object 
dtypes: float64(2), int64(5), object(7)
memory usage: 97.6+ KB


- 상관관계

In [70]:
train_numeric = preprocessed_train_df.select_dtypes(include='number')
train_numeric.corr()['Survived']
# 상관관계는 모두 크지 않아 보임

,Survived
PassengerId,-0.005007
Survived,1.000000
Pclass,-0.338481
Age,-0.077221
SibSp,-0.035322
Parch,0.081629
Fare,0.257307


In [18]:
# 모델에 사용할 특성(features)을 선정
# 제외해야 할 컬럼: "Ticket", "PassengerId", "Survived"
input_features = list(preprocessed_train_df.columns) # column list
input_features.remove("Ticket")
input_features.remove("PassengerId") # 단순 id로 불필요
#input_features.remove("Survived")    # survived와의 관계를 알아보는 것으로 상관계수가 1이므로 불필요

print(f"Input features: {input_features}")

Input features: ['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked', 'Ticket_number', 'Ticket_item']


In [19]:
input_features = list(preprocessed_test_df.columns) # column list
input_features.remove("Ticket")
input_features.remove("PassengerId")

In [20]:
# 이름을 토큰화 하는 함수
def tokenize_name(df):

  def extract_name(name):
    last_name = name.split(",")[0] if "," in name else "None"

    title_match = re.search(r",\s*([\w]+)\.", name)
    title = title_match.group(1) if title_match else "None"

    return last_name, title

  df[["Last_Name", "Title"]] = df["Name"].apply(lambda x : pd.Series(extract_name(x)))
  df.drop(columns=["Name"], inplace = True)

  return df

# 토큰화 적용
train = tokenize_name(preprocessed_train_df)
test = tokenize_name(preprocessed_test_df)

In [74]:
train.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Ticket_number,Ticket_item,Last_Name,Title
0,1,0,3,male,22.0,1,0,A/5 21171,7.2500,NaN,S,521171,A/5,Braund,Mr
1,2,1,1,female,38.0,1,0,PC 17599,71.2833,C85,C,17599,PC,Cumings,Mrs
2,3,1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,23101282,STON/O2.,Heikkinen,Miss
3,4,1,1,female,35.0,1,0,113803,53.1000,C123,S,113803,NONE,Futrelle,Mrs
4,5,0,3,male,35.0,0,0,373450,8.0500,NaN,S,373450,NONE,Allen,Mr


In [75]:
test.head()

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Ticket_number,Ticket_item,Last_Name,Title
0,892,3,male,34.5,0,0,330911,7.8292,NaN,Q,330911,NONE,Kelly,Mr
1,893,3,female,47.0,1,0,363272,7.0000,NaN,S,363272,NONE,Wilkes,Mrs
2,894,2,male,62.0,0,0,240276,9.6875,NaN,Q,240276,NONE,Myles,Mr
3,895,3,male,27.0,0,0,315154,8.6625,NaN,S,315154,NONE,Wirz,Mr
4,896,3,female,22.0,1,1,3101298,12.2875,NaN,S,3101298,NONE,Hirvonen,Mrs


In [21]:
train.drop(columns=['Cabin'],inplace=True)
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PassengerId    891 non-null    int64  
 1   Survived       891 non-null    int64  
 2   Pclass         891 non-null    int64  
 3   Sex            891 non-null    object 
 4   Age            714 non-null    float64
 5   SibSp          891 non-null    int64  
 6   Parch          891 non-null    int64  
 7   Ticket         891 non-null    object 
 8   Fare           891 non-null    float64
 9   Embarked       889 non-null    object 
 10  Ticket_number  891 non-null    object 
 11  Ticket_item    891 non-null    object 
 12  Last_Name      891 non-null    object 
 13  Title          891 non-null    object 
dtypes: float64(2), int64(5), object(7)
memory usage: 97.6+ KB


In [22]:
test.drop(columns=['Cabin'],inplace=True)
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PassengerId    418 non-null    int64  
 1   Pclass         418 non-null    int64  
 2   Sex            418 non-null    object 
 3   Age            332 non-null    float64
 4   SibSp          418 non-null    int64  
 5   Parch          418 non-null    int64  
 6   Ticket         418 non-null    object 
 7   Fare           417 non-null    float64
 8   Embarked       418 non-null    object 
 9   Ticket_number  418 non-null    object 
 10  Ticket_item    418 non-null    object 
 11  Last_Name      418 non-null    object 
 12  Title          418 non-null    object 
dtypes: float64(2), int64(4), object(7)
memory usage: 42.6+ KB


In [23]:
# 결측치 처리
value = train["Age"].mean()
train["Age"] = train["Age"].fillna(value)

In [24]:
value = test["Age"].mean()
test["Age"] = test["Age"].fillna(value)

In [25]:
train = train.dropna(subset=["Embarked"])
test = test.dropna(subset=["Fare"])

In [81]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PassengerId    889 non-null    int64  
 1   Survived       889 non-null    int64  
 2   Pclass         889 non-null    int64  
 3   Sex            889 non-null    object 
 4   Age            889 non-null    float64
 5   SibSp          889 non-null    int64  
 6   Parch          889 non-null    int64  
 7   Ticket         889 non-null    object 
 8   Fare           889 non-null    float64
 9   Embarked       889 non-null    object 
 10  Ticket_number  889 non-null    object 
 11  Ticket_item    889 non-null    object 
 12  Last_Name      889 non-null    object 
 13  Title          889 non-null    object 
dtypes: float64(2), int64(5), object(7)
memory usage: 104.2+ KB


In [82]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 417 entries, 0 to 417
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PassengerId    417 non-null    int64  
 1   Pclass         417 non-null    int64  
 2   Sex            417 non-null    object 
 3   Age            417 non-null    float64
 4   SibSp          417 non-null    int64  
 5   Parch          417 non-null    int64  
 6   Ticket         417 non-null    object 
 7   Fare           417 non-null    float64
 8   Embarked       417 non-null    object 
 9   Ticket_number  417 non-null    object 
 10  Ticket_item    417 non-null    object 
 11  Last_Name      417 non-null    object 
 12  Title          417 non-null    object 
dtypes: float64(2), int64(4), object(7)
memory usage: 45.6+ KB


## 3. 모델학습 및 평가

In [26]:
# 데이터 분할
X_data = train.drop(columns=['Survived'])
y_data = train['Survived']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_data,
                                                    y_data,
                                                    test_size=0.2)

In [27]:
# 모든 범주형 컬럼을 One-Hot Encoding
X_train = pd.get_dummies(X_train, drop_first=True)


In [45]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# 가능한 파라미터 조합 설정
params = {
    'n_estimators': [50, 100, 300, 500],
    'max_depth': [3, 4, 5, 10],
    'min_samples_split': [2, 5, 10],
    'max_features' : ["sqrt", "log2", None],
    'learning_rate' : [0.1,0.2,0.3]

}

gb = GradientBoostingClassifier(random_state=0)
gs = GridSearchCV(gb, param_grid = params, cv=5)

gs.fit(X_train, y_train)

# 최적의 파라미터 출력
print('최적의 파라미터', gs.best_params_)

# 최적의 스코어 출력
print('최적의 스코어', gs.best_score_)

# 최적의 모델 출력
print('최적의 모델', gs.best_estimator_)

In [46]:
X_train.columns

Index(['PassengerId', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male',
       'Ticket_110413', 'Ticket_110465', 'Ticket_110564',
       ...
       'Title_Major', 'Title_Master', 'Title_Miss', 'Title_Mlle', 'Title_Mme',
       'Title_Mr', 'Title_Mrs', 'Title_Ms', 'Title_None', 'Title_Rev'],
      dtype='object', length=1737)

In [43]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier

# 원본 test.csv 불러오기
test_data = pd.read_csv("test.csv")

# PassengerId 따로 저장
passenger_ids = test_data["PassengerId"]

# One-Hot Encoding
X_train = pd.get_dummies(X_train, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

# test 컬럼 맞추기
missing_cols = set(X_train.columns) - set(test.columns)
extra_cols = set(test.columns) - set(X_train.columns)

# 없는 컬럼 0으로 추가
missing_df = pd.DataFrame(0, index=test.index, columns=list(missing_cols))
test = pd.concat([test, missing_df], axis=1)

# test에만 있는 컬럼 삭제 후 컬럼 순서 맞추기
test = test.drop(columns=extra_cols, errors="ignore")[X_train.columns].copy()

# 인덱스 재정렬 (누락된 행이 있다면 복원)
test = test.reindex(test_data.index, fill_value=0)

# 나온 모델 학습
gb = GradientBoostingClassifier(
    learning_rate=0.3, max_depth=10, max_features='sqrt',
    min_samples_split=10, n_estimators=300, random_state=0
)
gb.fit(X_train, y_train)

# 예측
predictions = gb.predict(test)

# 제출 파일 생성
submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)

print("제출 파일 생성 완료: submission.csv")


제출 파일 생성 완료: submission.csv
